In [ ]:
from vislearnlabpy.embeddings.generate_embeddings import EmbeddingGenerator, EmbeddingConfig
from vislearnlabpy.embeddings.embedding_store import EmbeddingStore
from vislearnlabpy.embeddings.similarity_utils import text_image_sims_from_stores, csv_to_text_pairs, plot_rdm
import os
import numpy as np

## VisVocab testing
An example of using the embedding store to get similarity scores between embeddings. 

In [ ]:
config = EmbeddingConfig(device="cpu", output_type="doc")
generator = EmbeddingGenerator(config=config)
PROJECT_PATH = '/Users/vislearnlab/Documents/visual-precision'
# path to the csv with the trial metadata
input_csv = os.path.join(
                PROJECT_PATH, "data", "metadata", "level-imagepair_data.csv"
            )
# path to the images
input_path = os.path.join(PROJECT_PATH, "stimuli", "lookit", "exp1", "img")
specific_sim_pairs = csv_to_text_pairs(input_csv)

In [ ]:
#generator.generate_image_embeddings(output_path="visvocab", input_csv=input_csv, input_dir=input_path, batch_size=100, id_column=None)
image_embedding_store = EmbeddingStore.from_doc("visvocab/image_embeddings/clip_image_embeddings_doc.docs")
text_embedding_store = EmbeddingStore.from_doc("visvocab/text_embeddings/clip_text_embeddings_doc.docs")

Checking if embeddings look good.

In [ ]:
docs, scores = image_embedding_store.search_store(text_query="turkey")
print(docs.text)
print(scores)

Retrieving cosine similarities by default.

In [ ]:
text_image_sims_from_stores(text_embedding_store, image_embedding_store, "output/sims/similarities.csv", specific_sim_pairs)

RDM

In [ ]:
plot_rdm("output/sims", np.stack(image_embedding_store.EmbeddingList.embedding))

In [ ]:
bv_generator = EmbeddingGenerator.from_model("dinov3-babyview", device="cpu", output_type="doc")
bv_generator.generate_image_embeddings(output_path="visvocab", input_csv=input_csv, input_dir=input_path, batch_size=100, id_column=None)

In [ ]:
bv_image_embedding_store = EmbeddingStore.from_doc("visvocab/image_embeddings/dinov3-bv_image_embeddings_doc.docs")

In [ ]:
bv_image_embedding_store.retrieve_similarities(output_path="output/sims/bv_similarities.csv", text_pairs=specific_sim_pairs)

In [ ]:
pd.read_csv("output/sims/bv_similarities.csv")['cosine_similarity'].hist() 

In [ ]:
dino_v3_generator = EmbeddingGenerator.from_model("dinov3", device="cpu", output_type="doc")
dino_v3_generator.generate_image_embeddings(output_path="visvocab", input_csv=input_csv, input_dir=input_path, batch_size=100, id_column=None)
dinov3_base_generator = EmbeddingGenerator.from_model("dinov3-base", device="cpu", output_type="doc")
dinov3_base_generator.generate_image_embeddings(output_path="visvocab", input_csv=input_csv, input_dir=input_path, batch_size=100, id_column=None)

In [ ]:
from vislearnlabpy.embeddings.similarity_utils import compute_rdm, correlate_rdms
dinov3_image_embedding_store = EmbeddingStore.from_doc("visvocab/image_embeddings/dinov3-vitl16_image_embeddings_doc.docs")
dinov3_base_image_embedding_store = EmbeddingStore.from_doc("visvocab/image_embeddings/dinov3-vitb16_image_embeddings_doc.docs")
bv_rdm = compute_rdm(np.stack(bv_image_embedding_store.EmbeddingList.embedding), method="cosine")
clip_rdm = compute_rdm(np.stack(image_embedding_store.EmbeddingList.embedding), method="cosine")
dino_rdm = compute_rdm(np.stack(dinov3_image_embedding_store.EmbeddingList.embedding), method="cosine")
dino_base_rdm = compute_rdm(np.stack(dinov3_base_image_embedding_store.EmbeddingList.embedding), method="cosine")

In [ ]:
dinov3_base_image_embedding_store.EmbeddingList[0].embedding.shape

In [ ]:
dinov3_image_embedding_store.retrieve_similarities(output_path="output/sims/dino_similarities.csv", text_pairs=specific_sim_pairs)

In [ ]:
pd.read_csv("output/sims/dino_similarities.csv")['cosine_similarity'].hist() 

In [ ]:
saycam_generator = EmbeddingGenerator.from_model("dino_say_vitl16", device="cpu", output_type="doc")
saycam_generator.model.supports_text

In [ ]:
saycam_generator = EmbeddingGenerator.from_model("dino_say_vitl16", device="cpu", output_type="doc")
s_generator = EmbeddingGenerator.from_model("dino_s_vitl16", device="cpu", output_type="doc")
dinoimagenet_generator = EmbeddingGenerator.from_model("dino_imagenet100_vitb14", device="cpu", output_type="doc")
saycam_generator.generate_image_embeddings(output_path="visvocab", input_csv=input_csv, input_dir=input_path, batch_size=100, id_column=None)
s_generator.generate_image_embeddings(output_path="visvocab", input_csv=input_csv, input_dir=input_path, batch_size=100, id_column=None)
dinoimagenet_generator.generate_image_embeddings(output_path="visvocab", input_csv=input_csv, input_dir=input_path, batch_size=100, id_column=None)

In [ ]:
saycam_embedding_store = EmbeddingStore.from_doc("visvocab/image_embeddings/dino_say_vitl16_image_embeddings_doc.docs")
s_embedding_store = EmbeddingStore.from_doc("visvocab/image_embeddings/dino_s_vitl16_image_embeddings_doc.docs")
dinoimagenet_embedding_store = EmbeddingStore.from_doc("visvocab/image_embeddings/dino_imagenet100_vitb14_image_embeddings_doc.docs")

In [ ]:
saycam_rdm = compute_rdm(np.stack(saycam_embedding_store.EmbeddingList.embedding), method="cosine")
s_rdm = compute_rdm(np.stack(s_embedding_store.EmbeddingList.embedding), method="cosine")
dinoimagenet_rdm = compute_rdm(np.stack(dinoimagenet_embedding_store.EmbeddingList.embedding), method="cosine")

In [ ]:
print(f"Correlation between BV-DINO and CLIP: {correlate_rdms(bv_rdm, clip_rdm, correlation='spearman')}")
print(f"Correlation between BV-DINO and DINOv3: {correlate_rdms(bv_rdm, dino_rdm, correlation='spearman')}")
print(f"Correlation between CLIP and DINOv3: {correlate_rdms(clip_rdm, dino_rdm, correlation='spearman')}")
print(f"Correlation between Saycam-DINO and CLIP: {correlate_rdms(saycam_rdm, clip_rdm, correlation='spearman')}")
print(f"Correlation between Saycam-DINO and BV-DINO: {correlate_rdms(saycam_rdm, bv_rdm, correlation='spearman')}")
print(f"Correlation between Saycam-DINO and DINOv3: {correlate_rdms(saycam_rdm, dino_rdm, correlation='spearman')}")
print(f"Correlation between Saycam-DINO and DINO-Imagenet: {correlate_rdms(saycam_rdm, dinoimagenet_rdm, correlation='spearman')}") 
print(f"Correlation between DINO-Imagenet and CLIP: {correlate_rdms(dinoimagenet_rdm, clip_rdm, correlation='spearman')}")
print(f"Correlation between DINO-Imagenet and DINOv3: {correlate_rdms(dinoimagenet_rdm, dino_rdm, correlation='spearman')}")
print(f"Correlation between S-DINO and Saycam-DINO: {correlate_rdms(s_rdm, saycam_rdm, correlation='spearman')}")
print(f"Correlation between DINOl and DINOb: {correlate_rdms(dino_rdm, dino_base_rdm, correlation='spearman')}")

In [ ]:
import pandas as pd
sorted_df = pd.read_csv(input_csv).sort_values("animacy", ascending=False)  
df_vals = (
    pd.concat([
        sorted_df[["text1", "animacy"]].rename(columns={"text1": "text"}),
        sorted_df[["text2", "animacy"]].rename(columns={"text2": "text"}),
    ])
    .drop_duplicates(subset="text")
    .sort_values("animacy", ascending=False)["text"]
    .tolist()
)
df_vals

In [ ]:
image_embedding_store.compute_text_rdm(output_path="output_clip", order=df_vals)
dinov3_image_embedding_store.compute_text_rdm(output_path="output_dino", order=df_vals)
bv_image_embedding_store.compute_text_rdm(output_path="output_bv", order=df_vals)
saycam_embedding_store.compute_text_rdm(output_path="output_saycam", order=df_vals)


In [ ]:
dinov3_image_embedding_store.compute_text_rdm(output_path="output_dino", order=df_vals)


In [ ]:
dinoimagenet_embedding_store.compute_text_rdm(output_path="output_dinoimagenet", order=df_vals)

In [ ]:
dino_v3_generator.generate_image_embeddings(output_path="visvocab_dir", input_dir="../input", batch_size=100, id_column=None)

In [ ]:
dinov3_store = EmbeddingStore.from_doc("visvocab_dir/image_embeddings/dinov3-vitl16_image_embeddings_doc.docs")

In [ ]:
dinov3_store.EmbeddingList[5].url

In [ ]:
cat1_embedding = dinov3_store.EmbeddingList[4].embedding
bird_embedding = dinov3_store.EmbeddingList[1].embedding
cat2_embedding = dinov3_store.EmbeddingList[5].embedding
cosine_sim_cat1_cat2 = np.dot(cat1_embedding, cat2_embedding) / (np.linalg.norm(cat1_embedding) * np.linalg.norm(cat2_embedding))
print(f"Cosine similarity between cat1 and cat2: {cosine_sim_cat1_cat2}")   
cosine_sim_bird_cat = np.dot(bird_embedding, cat1_embedding) / (np.linalg.norm(bird_embedding) * np.linalg.norm(cat1_embedding))
print(f"Cosine similarity between bird and cat1: {cosine_sim_bird_cat}")

In [ ]:
from transformers import AutoModel, AutoImageProcessor
from PIL import Image
import torch
from torch.nn.functional import cosine_similarity

model_name = "facebook/dinov3-vitl16-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).eval()

cat1 = Image.open('/Users/vislearnlab/Documents/vllpy/examples/input/eyetrack1.png').convert("RGB")
cat2 = Image.open('/Users/vislearnlab/Documents/vllpy/examples/input/eyetrack2.png').convert("RGB")
bird = Image.open('/Users/vislearnlab/Documents/vllpy/examples/input/bird.png').convert("RGB")

with torch.no_grad():
    e_cat1 = model(**processor(images=cat1, return_tensors="pt")).pooler_output
    e_cat2 = model(**processor(images=cat2, return_tensors="pt")).pooler_output
    e_bird = model(**processor(images=bird, return_tensors="pt")).pooler_output

print(cosine_similarity(e_cat1, e_cat2))
print(cosine_similarity(e_cat1, e_bird))

In [ ]:
print(model.config.model_type)
print(model.config.hidden_size)
print(e_cat1.shape)
print(e_cat1[:5])  # first few values — are they all near-zero? all the same?

In [ ]:
print(e_cat1)
print(e_cat2)
print(e_cat1.shape, e_cat2.shape)

In [ ]:
from transformers import AutoModel, AutoImageProcessor

model_v2 = AutoModel.from_pretrained("facebook/dinov2-large").eval()
proc_v2 = AutoImageProcessor.from_pretrained("facebook/dinov2-large")

with torch.no_grad():
    e2_cat1 = model_v2(**proc_v2(images=cat1, return_tensors="pt")).pooler_output
    e2_cat2 = model_v2(**proc_v2(images=cat2, return_tensors="pt")).pooler_output
    e2_bird = model_v2(**proc_v2(images=bird, return_tensors="pt")).pooler_output

print(cosine_similarity(e2_cat1, e2_cat2))
print(cosine_similarity(e2_cat1, e2_bird))

In [ ]:
df_vals